# Urban Air Quality & Traffic Analysis
This notebook reads the gold-layer Parquet from S3, visualises key patterns, and trains the AQI prediction model.

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import sys
sys.path.append('../src')

import awswrangler as wr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

In [ ]:
# ── Load config & data ─────────────────────────────────────────────────────
with open('../config/aws_config.yaml') as f:
    cfg = yaml.safe_load(f)

BUCKET = cfg['s3']['bucket_name']
GOLD   = cfg['s3']['gold_prefix']

df = wr.s3.read_parquet(f's3://{BUCKET}/{GOLD}aqi_traffic_joined/')
print(f'Loaded {len(df):,} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# ── Basic stats ────────────────────────────────────────────────────────────
df.describe()

In [ ]:
# ── Plot 1: Average AQI by hour of day ────────────────────────────────────
hourly = df.groupby('hour')[['avg_aqi', 'avg_congestion_index']].mean().reset_index()

fig, ax1 = plt.subplots()
ax2 = ax1.twinx()

ax1.bar(hourly['hour'], hourly['avg_aqi'], color='steelblue', alpha=0.6, label='Avg AQI')
ax2.plot(hourly['hour'], hourly['avg_congestion_index'], 'r-o', label='Congestion Index')

ax1.set_xlabel('Hour of Day')
ax1.set_ylabel('Average AQI', color='steelblue')
ax2.set_ylabel('Congestion Index', color='red')
plt.title('AQI vs Traffic Congestion by Hour')
fig.tight_layout()
plt.savefig('aqi_vs_congestion_by_hour.png', dpi=150)
plt.show()

In [ ]:
# ── Plot 2: Rush hour vs Non-Rush hour AQI comparison ─────────────────────
rush = df.groupby('is_rush_hour')['avg_aqi'].mean()
rush.index = ['Non-Rush Hour', 'Rush Hour']

rush.plot(kind='bar', color=['#4CAF50', '#F44336'], edgecolor='black')
plt.title('Average AQI: Rush Hour vs Non-Rush Hour')
plt.ylabel('Average AQI')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('rush_hour_aqi_comparison.png', dpi=150)
plt.show()

In [ ]:
# ── Plot 3: Scatter — Congestion vs AQI ───────────────────────────────────
sns.scatterplot(data=df, x='avg_congestion_index', y='avg_aqi',
                hue='is_rush_hour', alpha=0.5)
plt.title('Congestion Index vs AQI')
plt.xlabel('Congestion Index (0=free flow, 1=gridlock)')
plt.ylabel('Average AQI')
plt.tight_layout()
plt.savefig('congestion_vs_aqi_scatter.png', dpi=150)
plt.show()

In [ ]:
# ── Correlation heatmap ────────────────────────────────────────────────────
num_cols = ['avg_aqi', 'avg_pm25', 'avg_speed_kmph', 'avg_congestion_index', 'hour']
corr = df[num_cols].corr()

sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
# ── Train & evaluate the Linear Regression model ──────────────────────────
from models.aqi_prediction_model import engineer_features, train_and_evaluate, save_model

X, y, feat_cols = engineer_features(df)
model, scaler, y_pred, y_test = train_and_evaluate(X, y)
save_model(model, scaler, feat_cols)

In [ ]:
# ── Plot 4: Predicted vs Actual AQI ───────────────────────────────────────
plt.scatter(y_test, y_pred, alpha=0.4, color='darkorange')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, 'k--', lw=1.5, label='Perfect prediction')
plt.xlabel('Actual AQI')
plt.ylabel('Predicted AQI')
plt.title('Linear Regression: Actual vs Predicted AQI')
plt.legend()
plt.tight_layout()
plt.savefig('actual_vs_predicted_aqi.png', dpi=150)
plt.show()